# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [9]:
import csv


def transpose(M):
    return [list(row) for row in zip(*M)]


def matmul(A, B):
    bt = transpose(B)
    return [[sum(a * b for a, b in zip(row, col)) for col in bt] for row in A]


def matadd(A, B):
    return [[a + b for a, b in zip(ra, rb)] for ra, rb in zip(A, B)]


def matsub(A, B):
    return [[a - b for a, b in zip(ra, rb)] for ra, rb in zip(A, B)]


def eye(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def inverse(M):
    n = len(M)
    aug = [row[:] + eye(n)[i] for i, row in enumerate(M)]
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(aug[r][col]))
        if abs(aug[pivot][col]) < 1e-15:
            raise ValueError("Singular matrix")
        aug[col], aug[pivot] = aug[pivot], aug[col]
        pv = aug[col][col]
        aug[col] = [v / pv for v in aug[col]]
        for r in range(n):
            if r == col:
                continue
            f = aug[r][col]
            aug[r] = [rv - f * cv for rv, cv in zip(aug[r], aug[col])]
    return [row[n:] for row in aug]


def max_abs_diff(A, B):
    return max(abs(a - b) for ra, rb in zip(A, B) for a, b in zip(ra, rb))


def read_matrix_tsv(path):
    with open(path, newline="") as f:
        reader = csv.reader(f, delimiter="	")
        header = next(reader)
        col_labels = header[1:]
        row_labels = []
        data = []
        for row in reader:
            row_labels.append(row[0])
            data.append([float(v) for v in row[1:]])
    return row_labels, col_labels, data


_, _, A = read_matrix_tsv("model-A.tsv")
_, _, B = read_matrix_tsv("model-B.tsv")
_, _, Q = read_matrix_tsv("cost-Q.tsv")
_, _, R = read_matrix_tsv("cost-R.tsv")

At = transpose(A)
Bt = transpose(B)
P = [row[:] for row in Q]

for _ in range(10000):
    BtP = matmul(Bt, P)
    inv_term = inverse(matadd(R, matmul(BtP, B)))
    K = matmul(matmul(inv_term, BtP), A)
    P_next = matadd(
        Q,
        matsub(
            matmul(matmul(At, P), A),
            matmul(matmul(matmul(At, P), B), K),
        ),
    )
    if max_abs_diff(P_next, P) < 1e-12:
        P = P_next
        break
    P = P_next

K = matmul(matmul(inverse(matadd(R, matmul(matmul(Bt, P), B))), matmul(Bt, P)), A)
K


[[0.33445985227224767, 1.3044541296642804, 1.8581308846100588]]

Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [10]:
with open("control-K-intended.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="	")
    writer.writerow(["index", "x", "v", "a"])
    writer.writerow(["u"] + [f"{v:.15g}" for v in K[0]])


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [11]:
qc_check = []

with open("qc-log.tsv", newline="") as f:
    reader = csv.DictReader(f, delimiter="	")
    for row in reader:
        x = [float(row["x"]), float(row["v"]), float(row["a"])]
        u_check = -sum(K[0][j] * x[j] for j in range(3))
        qc_check.append((row["index"], u_check))

qc_check[:5]


[('0', 1.6722992613612384),
 ('1', -1.152095347507631),
 ('2', -1.2351825827057443),
 ('3', -0.2885035031677534),
 ('4', 0.36920157289926414)]

Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [12]:
with open("qc-check.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="	")
    writer.writerow(["index", "u_check"])
    for idx, u in qc_check:
        writer.writerow([idx, f"{u:.15g}"])


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [13]:
def least_squares(X, Y):
    Xt = transpose(X)
    XtX = matmul(Xt, X)
    XtY = matmul(Xt, Y)
    return matmul(inverse(XtX), XtY)


X = []
Y = []
with open("qc-log.tsv", newline="") as f:
    reader = csv.DictReader(f, delimiter="	")
    for row in reader:
        X.append([float(row["x"]), float(row["v"]), float(row["a"])])
        Y.append([float(row["u"])])

beta = least_squares(X, Y)  # u = X beta
K_actual = [[-beta[0][0], -beta[1][0], -beta[2][0]]]
K_actual


[[0.34043755006707427, 1.3001202346439198, 1.950116964661527]]

Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [14]:
with open("control-K-actual.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="	")
    writer.writerow(["index", "x", "v", "a"])
    writer.writerow(["u"] + [f"{v:.15g}" for v in K_actual[0]])


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [15]:
rows = []
with open("qc-train.tsv", newline="") as f:
    reader = csv.DictReader(f, delimiter="	")
    for row in reader:
        rows.append((
            float(row["x"]),
            float(row["v"]),
            float(row["a"]),
            float(row["u"]),
        ))

Phi = []
Y_next = []
for i in range(len(rows) - 1):
    x, v, a, u = rows[i]
    xn, vn, an, _ = rows[i + 1]
    Phi.append([x, v, a, u])
    Y_next.append([xn, vn, an])

Theta = least_squares(Phi, Y_next)  # shape 4x3
A_new = [
    [Theta[0][0], Theta[1][0], Theta[2][0]],
    [Theta[0][1], Theta[1][1], Theta[2][1]],
    [Theta[0][2], Theta[1][2], Theta[2][2]],
]
B_new = [[Theta[3][0]], [Theta[3][1]], [Theta[3][2]]]

A_new, B_new


([[1.000000000000008, 1.1000000000000352, 5.5067062021407764e-14],
  [5.329070518200751e-15, 0.9000000000000163, 0.9500000000000242],
  [2.220446049250313e-16, 0.0, 1.1000000000000032]],
 [[1.6431300764452317e-14], [-0.00999999999998824], [0.9000000000000012]])

Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [16]:
with open("model-A-new.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="	")
    writer.writerow(["index", "x", "v", "a"])
    for label, row in zip(["x", "v", "a"], A_new):
        writer.writerow([label] + [f"{v:.15g}" for v in row])

with open("model-B-new.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="	")
    writer.writerow(["index", "u"])
    for label, row in zip(["x", "v", "a"], B_new):
        writer.writerow([label, f"{row[0]:.15g}"])


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Collaboration: none.

Additional libraries: none.

Generative AI usage: ChatGPT was used for debugging support and code review. Transcript:
https://chatgpt.com/
